# Rock and target

**What you will learn:** reach a target while avoiding a rock, with a trade-off
parameter $\Lambda$ that weights target versus obstacle. Compare a running-cost
penalty (no native `pyspect` equivalent) with a level-set formulation.

**pyspect API:** `TVHJImpl`, `DriftingCanoeBall`, `impl.grid`, `impl.avoid_dynamics`
(reused with raw `hj.solve` where postprocessors are needed)

**Prerequisites:** `rock_and_eddy.ipynb` (penalty vs hard constraint)

A canoe drifts downstream and can push in any direction (`DriftingCanoeBall`).
A rock sits at the origin (radius 5), a target at $(0, -7)$ (radius 2).
Starting from $(0, 9)$, how does the reachable set change as $\Lambda$ shifts
priority from the target to the rock?


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from scipy.integrate import solve_ivp

import hj_reachability as hj
from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import DriftingCanoeBall


In [ ]:
T = 20
DOMAIN = [-10, 10]
x0 = np.array([0.0, 9.0])

def build(vox):
    axes = [dict(name='t', bounds=[0, T], points=2 * vox + 1),
            dict(name='x', bounds=DOMAIN, points=vox + 1),
            dict(name='y', bounds=DOMAIN, points=vox + 1)]
    impl = TVHJImpl(dict(cls=DriftingCanoeBall), axes, accuracy='very_high')
    return impl

def level_sets(impl):
    S = impl.grid.states
    rock = jnp.sqrt(S[..., 0] ** 2 + S[..., 1] ** 2) - 5.0
    target = -jnp.sqrt(S[..., 0] ** 2 + (S[..., 1] + 7.0) ** 2) + 2.0
    return rock, target


## The setting

Downstream drift, full 2D control, one rock (red) and one target (green).


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
th = np.linspace(0, 2 * np.pi, 100)
ax.fill(5 * np.cos(th), 5 * np.sin(th), color='salmon', alpha=0.7, label='Rock')
ax.fill(2 * np.cos(th), -7 + 2 * np.sin(th), color='lightgreen', alpha=0.7, label='Target')
gx, gy = np.meshgrid(np.linspace(-8, 8, 9), np.linspace(-8, 8, 9))
ax.quiver(gx, gy, 0.0 * gx, -1.0 + 0.0 * gy, color='b', alpha=0.5, label='Current')
ax.plot([x0[0]], [x0[1]], 'ko', markersize=8, label='Start')
ax.set_xlim(DOMAIN); ax.set_ylim(DOMAIN); ax.set_aspect('equal')
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title('Reach the green disc, stay outside the red disc')
ax.legend(loc='upper right', framealpha=1.0)
plt.show()


## Approach 1 - running-cost penalty

The penalty enters the Hamiltonian: $H + (1-\Lambda) r + \Lambda q$.
There is no `pyspect` wrapper for `hamiltonian_postprocessor`, so this branch
calls `hj.solve` on the dynamics and grid that `TVHJImpl` already built.


In [ ]:
def penalty_value(impl, Lambda):
    rock, target = level_sets(impl)
    q = jnp.where(rock <= 0.0, -1.0, 0.0)
    r = jnp.where(target > 0.0, 1.0, 0.0)
    settings = hj.SolverSettings.with_accuracy(
        'very_high',
        hamiltonian_postprocessor=lambda H: H + (1 - Lambda) * r + Lambda * q,
    )
    times = np.linspace(0.0, -T, len(impl.timeline))
    V = hj.solve(settings, impl.avoid_dynamics, impl.grid, times, 0.0 * r,
                 progress_bar=False)
    return np.asarray(V)


## Approach 2 - level-set reach-avoid with $\Lambda$

The value function is clamped each step:
$V \leftarrow \min(\max(V, \Lambda r), (1-\Lambda) q)$.
This uses `value_postprocessor`, which also has no `pyspect` equivalent; the
standard `impl.reach(l, g)` API covers hard target/constraint sets, not this
soft trade-off.


In [ ]:
def hjr_value(impl, Lambda):
    rock, target = level_sets(impl)
    q = rock
    r = target
    initial = jnp.minimum(Lambda * r, (1 - Lambda) * q)

    def value_postprocessor(t, V):
        return jnp.minimum(jnp.maximum(V, Lambda * r), (1 - Lambda) * q)

    settings = hj.SolverSettings.with_accuracy(
        'very_high', value_postprocessor=value_postprocessor)
    times = np.linspace(0.0, -T, len(impl.timeline))
    V = hj.solve(settings, impl.avoid_dynamics, impl.grid, times, initial,
                 progress_bar=False)
    return np.asarray(V)


## Closed-loop trajectory

Sample-and-hold optimal control from the value-function tube (same logic as
`closed_loop.py` in hjr_examples, inlined here).


In [ ]:
def closed_loop_trajectory(model, grid, times, V, initial_state, steps=40):
    times = np.asarray(times)
    V = np.asarray(V)
    if times[0] > times[-1]:  # hj.solve uses decreasing time; integrate forward
        times = times[::-1]
        V = V[::-1]

    us, sols = [], []
    state = np.asarray(initial_state, dtype=float)
    t0, t1 = times[0], times[-1]

    for i in range(steps):
        t = t0 + (t1 - t0) * i / steps
        t_plus = t0 + (t1 - t0) * (i + 1) / steps
        j = np.searchsorted(times, t, side='right') - 1
        k = min(j + 1, len(times) - 1)
        if j == k:
            grad = grid.interpolate(grid.grad_values(V[j]), state=state)
        else:
            gl = grid.interpolate(grid.grad_values(V[j]), state=state)
            gr = grid.interpolate(grid.grad_values(V[k]), state=state)
            grad = ((t - times[j]) * gr + (times[k] - t) * gl) / (times[k] - times[j])

        u = model.optimal_control(state, t, grad)
        d = model.optimal_disturbance(state, t, grad)

        def rhs(time, x, u=u, d=d):
            return np.array(model(x, u, d, time))

        sol = solve_ivp(rhs, [t, t_plus], state, dense_output=True)
        state = sol.sol(t_plus)
        us.append(u)
        sols.append(sol.sol)

    def x(t):
        if t <= t0:
            return sols[0](t0)
        if t >= t1:
            return sols[-1](t1)
        idx = int((t - t0) / (t1 - t0) * steps)
        idx = min(max(idx, 0), steps - 1)
        return sols[idx](t)

    return x


## Resolution and $\Lambda$ sweep

Same grid as hjr_examples: $\Lambda \in \{0.01, 0.5, 0.99\}$ and
$500 \times 500$, $100 \times 100$, $20 \times 20$ spatial resolutions.

Expect **a few minutes** for the full sweep (18 PDE solves,
including one at $500\times500$).


In [ ]:
voxes = [500, 100, 20]
Lambdas = [0.01, 0.5, 0.99]

Vs, HJR_Vs, cls, HJR_cls, grids = [], [], [], [], []

for vox in voxes:
    print(f'vox={vox}...')
    impl = build(vox)
    Vs.append([])
    HJR_Vs.append([])
    cls.append([])
    HJR_cls.append([])
    grids.append(impl.grid)
    times = np.linspace(0.0, -T, len(impl.timeline))
    for Lambda in Lambdas:
        V = penalty_value(impl, Lambda)
        Vs[-1].append(V[-1])
        cls[-1].append(closed_loop_trajectory(
            impl.avoid_dynamics, impl.grid, times,
            V, x0))

        Vh = hjr_value(impl, Lambda)
        HJR_Vs[-1].append(Vh[-1])
        HJR_cls[-1].append(closed_loop_trajectory(
            impl.avoid_dynamics, impl.grid, times,
            Vh, x0))
    print(f'  {vox}x{vox} done')


In [ ]:
def plot_panel(V, grid, cl, ax, level=0.0, show_title=False):
    V = np.clip(np.asarray(V), -10.0, 10.0)
    x = np.asarray(grid.states[:, 0, 0])
    y = np.asarray(grid.states[0, :, 1])
    X, Y = np.meshgrid(x, y)
    mesh = ax.pcolormesh(Y, X, V.T, cmap='viridis', vmin=-10, vmax=10, shading='auto')
    th = np.linspace(0, 2 * np.pi, 100)
    ax.plot(5 * np.cos(th), 5 * np.sin(th), color='red', lw=2)
    ax.plot(2 * np.cos(th), -7 + 2 * np.sin(th), color='green', lw=2)
    ax.contour(Y, X, V, levels=[level], colors='black', linestyles='--', lw=2)
    ts = np.linspace(-T, 0, 80)
    ax.plot([cl(t)[0] for t in ts], [cl(t)[1] for t in ts], color='white', lw=2.5)
    ax.set_aspect('equal')
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    if show_title:
        ax.set_title('$V(x,y,t=0)$')
    return mesh


def plot_grid(all_V, all_cls, all_grids, level, row_label):
    fig, axs = plt.subplots(3, 3, figsize=(14, 14))
    mesh = None
    for i, Lambda in enumerate(Lambdas):
        for j, vox in enumerate(voxes):
            mesh = plot_panel(all_V[j][i], all_grids[j], all_cls[j][i], axs[i, j],
                              level=level, show_title=(i == 0 and j == 1))
    fig.tight_layout(rect=[0.12, 0.02, 0.98, 0.90])
    for i, Lambda in enumerate(Lambdas):
        pos = axs[i, 0].get_position()
        fig.text(0.07, (pos.y0 + pos.y1) / 2, row_label + f' $= {Lambda}$',
                 ha='center', va='center', fontsize=22, fontweight='bold',
                 rotation=90, transform=fig.transFigure)
    for j, vox in enumerate(voxes):
        pos = axs[0, j].get_position()
        fig.text((pos.x0 + pos.x1) / 2, 0.935, f'${vox}\times{vox}$',
                 ha='center', va='center', fontsize=22, fontweight='bold',
                 transform=fig.transFigure)
    left_pos, right_pos = axs[0, 0].get_position(), axs[0, 2].get_position()
    fig.add_artist(FancyArrowPatch(
        posA=(left_pos.x0, 0.96), posB=(right_pos.x1, 0.96),
        transform=fig.transFigure, arrowstyle='->', mutation_scale=30, lw=2.5))
    fig.text((left_pos.x0 + right_pos.x1) / 2, 0.975, 'Decreasing resolution',
             ha='center', va='center', fontsize=24, fontweight='bold',
             transform=fig.transFigure)
    top_pos, bot_pos = axs[0, 0].get_position(), axs[2, 0].get_position()
    fig.add_artist(FancyArrowPatch(
        posA=(0.025, top_pos.y1), posB=(0.025, bot_pos.y0),
        transform=fig.transFigure, arrowstyle='->', mutation_scale=30, lw=2.5))
    fig.text(0.012, (top_pos.y1 + bot_pos.y0) / 2, 'Increasing ' + row_label,
             ha='center', va='center', fontsize=24, fontweight='bold',
             rotation=90, transform=fig.transFigure)
    fig.colorbar(mesh, ax=axs.ravel().tolist(), label='Value', fraction=0.02, pad=0.02)
    plt.show()

# Penalty contours at 0.01, HJR at 0.0 (as in hjr_examples)
plot_grid(Vs, cls, grids, level=0.01, row_label=r'$\Lambda$')
plot_grid(HJR_Vs, HJR_cls, grids, level=0.0, row_label=r'$\Lambda$')
